In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# ---------------- SEQUENCE CREATION ----------------
def create_seq(X, y, seq_len=25):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)


# ---------------- METRICS ----------------
def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true,  y_pred)
    mape = np.mean(np.abs((y_true - y_pred)/y_true))*100

    print(f"\n===== {label} =====")
    print("MAE :",  round(mae,3))
    print("RMSE:",  round(rmse,3))
    print("R2  :",  round(r2,3))
    print("MAPE:", round(mape,2), "%")

    return mae, rmse, r2, mape


# ---------------- BUILD LSTM MODEL ----------------
def build_lstm(input_shape):
    model = Sequential([
        LSTM(96, return_sequences=True, input_shape=input_shape),
        Dropout(0.25),

        LSTM(64, return_sequences=False),
        Dropout(0.25),

        Dense(32, activation='relu'),
        Dense(2)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model


# ---------------- MAIN FUNCTION ----------------
def run_LSTM(df, name):

    print(f"\n================ LSTM for {name} ================")

    # Features and target
    X = df[['Open','High','Low','Close']].values
    y = df[['High','Low']].values

    seq_len = 25
    train_size = max(int(len(df)*0.85), seq_len+3)
    train_size = min(train_size, len(df)-2)

    X_train_raw = X[:train_size]
    X_test_raw  = X[train_size:]

    y_train_raw = y[:train_size]
    y_test_raw  = y[train_size:]

    # Scaling
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train_raw)
    X_test_scaled  = scaler_X.transform(X_test_raw)

    y_train_scaled = scaler_y.fit_transform(y_train_raw)
    y_test_scaled  = scaler_y.transform(y_test_raw)

    X_scaled = np.vstack([X_train_scaled, X_test_scaled])
    y_scaled = np.vstack([y_train_scaled, y_test_scaled])

    # Sequence creation
    X_seq, y_seq = create_seq(X_scaled, y_scaled, seq_len)

    split = train_size - seq_len
    X_train, X_test = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]

    # Adaptive batch size
    batch = 16 if len(X_train)>50 else 4

    model = build_lstm((seq_len, X_train.shape[2]))

    es = EarlyStopping(patience=6, restore_best_weights=True)
    lr = ReduceLROnPlateau(factor=0.5, patience=3)

    model.fit(
        X_train, y_train,
        epochs=60,
        batch_size=batch,
        validation_data=(X_test, y_test) if len(X_test)>1 else None,
        callbacks=[es, lr],
        verbose=1
    )

    # Prediction
    pred_scaled = model.predict(X_test)
    pred = scaler_y.inverse_transform(pred_scaled)
    y_true = scaler_y.inverse_transform(y_test)

    pred_high, pred_low = pred[:,0], pred[:,1]
    true_high, true_low = y_true[:,0], y_true[:,1]

    m1 = metrics(true_high, pred_high, f"LSTM {name} High")
    m2 = metrics(true_low, pred_low, f"LSTM {name} Low")

    return (m1, m2)


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [3]:
df = pd.read_csv("crude_oil_2007_to_present.csv")
df = df.rename(columns={df.columns[0]: "Date"})
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

for col in df.columns:
    if col != "Date":
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(",","",regex=False), errors="coerce")

df = df.sort_values("Date").fillna(method="ffill").fillna(method="bfill")
df = df[["Date","Close"]].dropna().reset_index(drop=True)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_27748\2959239312.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
C:\Users\ASUS\AppData\Local\Temp\ipykernel_27748\2959239312.py:9: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.sort_values("Date").fillna(method="ffill").fillna(method="bfill")


In [4]:
df1 = df[df["Date"].dt.year <= 2011]
df2 = df[(df["Date"].dt.year > 2011) & (df["Date"].dt.year <= 2019)]
df3 = df[(df["Date"].dt.year >= 2020) & (df["Date"].dt.year <= 2022)]
df4 = df[df["Date"].dt.year > 2022]

splits = [
    ("2007–2011", df1),
    ("2011–2019", df2),
    ("2020–2022", df3),
    ("2022–Latest", df4)
]

In [10]:
import pandas as pd

df = pd.read_csv("crude_oil_2007_to_present.csv")
print(df.columns)


Index(['Price', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')


In [18]:
df.head()

,Price,Close,High,Low,Open,Volume
0,2007-01-03,58.31999969482422,60.970001220703125,57.720001220703125,60.95000076293945,248547
1,2007-01-04,55.59000015258789,58.58000183105469,55.38999938964844,58.349998474121094,324010
2,2007-01-05,56.310001373291016,56.400001525878906,54.900001525878906,55.40999984741211,250130
3,2007-01-08,56.09000015258789,57.720001220703125,55.099998474121094,56.2400016784668,298727
4,2007-01-09,55.63999938964844,56.20000076293945,53.880001068115234,56.119998931884766,333330


In [11]:
import pandas as pd
import numpy as np

# ---------------- LOAD DATA ----------------
df = pd.read_csv("crude_oil_2007_to_present.csv")

# The first column is actually the Date column, rename it properly
df = df.rename(columns={df.columns[0]: "Date"})

# Convert Date column to proper datetime
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Drop invalid rows if any occurred during shifting
df = df.dropna(subset=["Date"])

# Convert numeric columns safely
for col in df.columns:
    if col != "Date":
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Fill missing values created during conversion
df = df.ffill().bfill()

# Set date as index
df = df.set_index("Date")

print(df.head())
print(df.dtypes)

# ---------------- CREATE REGIME-BASED SPLITS ----------------
df1 = df.loc[: "2010-12-31"]              # Pre-2011
df2 = df.loc["2011-01-01":"2019-12-31"]   # 2011–2019
df3 = df.loc["2020-01-01":"2021-12-31"]   # COVID period
df4 = df.loc["2022-01-01":]               # 2022 onward

# Print sizes
print("\nSizes of each segment:")
print("df1 (2007–2010):", len(df1))
print("df2 (2011–2019):", len(df2))
print("df3 (2020–2021):", len(df3))
print("df4 (2022–present):", len(df4))


                Close       High        Low       Open  Volume
Date                                                          
2007-01-02  61.049999  61.549999  60.410000  60.980000   41686
2007-01-03  58.320000  60.970001  57.720001  60.950001  248547
2007-01-04  55.590000  58.580002  55.389999  58.349998  324010
2007-01-05  56.310001  56.400002  54.900002  55.410000  250130
2007-01-08  56.090000  57.720001  55.099998  56.240002  298727
Close     float64
High      float64
Low       float64
Open      float64
Volume      int64
dtype: object

Sizes of each segment:
df1 (2007–2010): 1009
df2 (2011–2019): 2262
df3 (2020–2021): 505
df4 (2022–present): 970


C:\Users\ASUS\AppData\Local\Temp\ipykernel_27748\51227745.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")


In [12]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Dictionary to store all results
results = {
    "XGBoost": {},
    "CatBoost": {},
    "TCN": {},
    "ARIMA": {},
    "SARIMA": {},
    "BiLSTM": {},
    "ConvLSTM": {}
}

datasets = {
    "df1": df1,
    "df2": df2,
    "df3": df3,
    "df4": df4
}

def prepare_data(df):
    df = df.copy()

 

    # Multi-output targets
    y = df[["High", "Low"]]           # predicting both
    X = df.drop(["High", "Low"], axis=1)

    return train_test_split(X, y, test_size=0.2, random_state=42)


In [14]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

def evaluate_model(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # Avoid divide-by-zero in MAPE
    non_zero_idx = y_true != 0
    mape = np.mean(np.abs((y_true[non_zero_idx] - y_pred[non_zero_idx]) / y_true[non_zero_idx])) * 100

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)

    return {
        "MAPE": mape,
        "RMSE": rmse,
        "R2": r2,
        "MAE": mae
    }


In [13]:
from xgboost import XGBRegressor

for name, df in datasets.items():
    X_train, X_test, y_train, y_test = prepare_data(df)

    model = XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8
    )
    
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    results["XGBoost"][name] = {
        "RMSE": np.sqrt(mean_squared_error(y_test, preds)),
        "R2": r2_score(y_test, preds)
    }

results["XGBoost"]


{'df1': {'RMSE': 0.8763150142870038, 'R2': 0.9985418319702148},
 'df2': {'RMSE': 0.5251721065979592, 'R2': 0.9994839429855347},
 'df3': {'RMSE': 0.961987642678738, 'R2': 0.9975806474685669},
 'df4': {'RMSE': 0.7076436125506455, 'R2': 0.9967510104179382}}

In [ ]:
from xgboost import XGBRegressor

for name, df in datasets.items():
    X_train, X_test, y_train, y_test = prepare_data(df)

    model = XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    results["XGBoost"][name] = evaluate_model(y_test, preds)
    


In [16]:
results["XGBoost"]

{'df1': {'MAPE': 0.8174012148676921,
  'RMSE': 0.8672458704798015,
  'R2': 0.9985720171421495,
  'MAE': 0.6039343446788227},
 'df2': {'MAPE': 0.5795538088344682,
  'RMSE': 0.5067259183549351,
  'R2': 0.999519625229482,
  'MAE': 0.37688524991471206},
 'df3': {'MAPE': 2.0358431381482465,
  'RMSE': 1.2120794587351273,
  'R2': 0.9961733134714502,
  'MAE': 0.6637474971242471},
 'df4': {'MAPE': 0.5777154355572699,
  'RMSE': 0.6824713987566698,
  'R2': 0.9969895972992724,
  'MAE': 0.4671505603593648}}

In [18]:
from statsmodels.tsa.arima.model import ARIMA

for name, df in datasets.items():
    y = df["High"]  # choose High or Low separately

    train_size = int(len(y) * 0.8)
    train, test = y[:train_size], y[train_size:]

    model = ARIMA(train, order=(2,1,2))
    model_fit = model.fit()
    preds = model_fit.forecast(len(test))

    results["ARIMA"][name] = evaluate_model(test, preds)


d:\Anacondaa\envs\tfenv\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
d:\Anacondaa\envs\tfenv\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
d:\Anacondaa\envs\tfenv\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
d:\Anacondaa\envs\tfenv\lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
d:\Anaco

In [19]:
results["ARIMA"]

{'df1': {'MAPE': 5.728100384465621,
  'RMSE': 5.415116089870872,
  'R2': -0.0468336313252562,
  'MAE': 4.553519734868476},
 'df2': {'MAPE': 9.62455466964337,
  'RMSE': 6.887764450238442,
  'R2': -0.0034815063213464725,
  'MAE': 5.912673380525837},
 'df3': {'MAPE': 13.346350242940568,
  'RMSE': 12.183360717486662,
  'R2': -3.356772925104445,
  'MAE': 10.491911458600928},
 'df4': {'MAPE': 12.30538878154422,
  'RMSE': 8.660861366932698,
  'R2': -3.3546272743415777,
  'MAE': 7.8399596457817395}}

In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# ---------------- SEQUENCE CREATION ----------------
def create_seq(X, y, seq_len=25):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)


# ---------------- METRICS ----------------
def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true,  y_pred)
    mape = np.mean(np.abs((y_true - y_pred)/y_true))*100

    print(f"\n===== {label} =====")
    print("MAE :",  round(mae,3))
    print("RMSE:",  round(rmse,3))
    print("R2  :",  round(r2,3))
    print("MAPE:", round(mape,2), "%")

    return mae, rmse, r2, mape


# ---------------- BUILD BiLSTM MODEL ----------------
def build_bilstm(input_shape):
    model = Sequential([

        Bidirectional(LSTM(96, return_sequences=True), input_shape=input_shape),
        Dropout(0.25),

        Bidirectional(LSTM(64, return_sequences=False)),
        Dropout(0.25),

        Dense(32, activation='relu'),
        Dense(2)
    ])

    model.compile(optimizer='adam', loss='mse')
    return model


# ---------------- MAIN FUNCTION ----------------
def run_BiLSTM(df, name):

    print(f"\n================ BiLSTM for {name} ================")

    # Features and target
    X = df[['Open','High','Low','Close']].values
    y = df[['High','Low']].values

    seq_len = 25
    train_size = max(int(len(df)*0.85), seq_len+3)
    train_size = min(train_size, len(df)-2)

    X_train_raw = X[:train_size]
    X_test_raw  = X[train_size:]

    y_train_raw = y[:train_size]
    y_test_raw  = y[train_size:]

    # Scaling
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train_raw)
    X_test_scaled  = scaler_X.transform(X_test_raw)

    y_train_scaled = scaler_y.fit_transform(y_train_raw)
    y_test_scaled  = scaler_y.transform(y_test_raw)

    X_scaled = np.vstack([X_train_scaled, X_test_scaled])
    y_scaled = np.vstack([y_train_scaled, y_test_scaled])

    # Sequence creation
    X_seq, y_seq = create_seq(X_scaled, y_scaled, seq_len)

    split = train_size - seq_len
    X_train, X_test = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]

    # Adaptive batch size
    batch = 16 if len(X_train)>50 else 4

    model = build_bilstm((seq_len, X_train.shape[2]))

    es = EarlyStopping(patience=6, restore_best_weights=True)
    lr = ReduceLROnPlateau(factor=0.5, patience=3)

    model.fit(
        X_train, y_train,
        epochs=60,
        batch_size=batch,
        validation_data=(X_test, y_test) if len(X_test)>1 else None,
        callbacks=[es, lr],
        verbose=1
    )

    # Prediction
    pred_scaled = model.predict(X_test)
    pred = scaler_y.inverse_transform(pred_scaled)
    y_true = scaler_y.inverse_transform(y_test)

    pred_high, pred_low = pred[:,0], pred[:,1]
    true_high, true_low = y_true[:,0], y_true[:,1]

    m1 = metrics(true_high, pred_high, f"BiLSTM {name} High")
    m2 = metrics(true_low,  pred_low,  f"BiLSTM {name} Low")

    return (m1, m2)


In [26]:
# ================= RUN BILSTM ON ALL DATASETS ================= #

# df1, df2, df3, df4 must already be loaded before running this cell.

print("\n\n====== RUNNING BiLSTM ON ALL DATASETS ======\n")

results_bilstm = {}

for df, name in zip([df1, df2, df3, df4], ["DF1", "DF2", "DF3", "DF4"]):
    print(f"\n\n----- Training BiLSTM for {name} -----")
    try:
        res = run_BiLSTM(df, name)
        results_bilstm[name] = {
            "High": {
                "MAE":  res[0][0],
                "RMSE": res[0][1],
                "R2":   res[0][2],
                "MAPE": res[0][3]
            },
            "Low": {
                "MAE":  res[1][0],
                "RMSE": res[1][1],
                "R2":   res[1][2],
                "MAPE": res[1][3]
            }
        }
    except Exception as e:
        print(f"Error in {name}: {e}")

print("\n\n============= FINAL BiLSTM RESULTS =============\n")
results_bilstm




====== RUNNING BiLSTM ON ALL DATASETS ======



----- Training BiLSTM for DF1 -----

================ BiLSTM for DF1 ================


d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - loss: 0.1664 - val_loss: 0.0121 - learning_rate: 0.0010
Epoch 2/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0419 - val_loss: 0.0078 - learning_rate: 0.0010
Epoch 3/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0356 - val_loss: 0.0067 - learning_rate: 0.0010
Epoch 4/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0302 - val_loss: 0.0073 - learning_rate: 0.0010
Epoch 5/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0287 - val_loss: 0.0054 - learning_rate: 0.0010
Epoch 6/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0338 - val_loss: 0.0056 - learning_rate: 0.0010
Epoch 7/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0298 - val_loss: 0.0059 - learning_rate: 0.0010
Epoch 8/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0241 - val_loss: 0.0054 - learning_rate: 0.0010
Epoch 9/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0236 - val_loss: 0.0052 - learning_rate: 5.0000e-04
Epoch 10/60
52/52

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


119/119 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - loss: 0.0678 - val_loss: 0.0114 - learning_rate: 0.0010
Epoch 2/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0229 - val_loss: 0.0071 - learning_rate: 0.0010
Epoch 3/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0187 - val_loss: 0.0072 - learning_rate: 0.0010
Epoch 4/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0155 - val_loss: 0.0063 - learning_rate: 0.0010
Epoch 5/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0143 - val_loss: 0.0068 - learning_rate: 0.0010
Epoch 6/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0120 - val_loss: 0.0038 - learning_rate: 0.0010
Epoch 7/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0125 - val_loss: 0.0040 - learning_rate: 0.0010
Epoch 8/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0111 - val_loss: 0.0039 - learning_rate: 0.0010
Epoch 9/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0102 - val_loss: 0.0038 - learning_rate: 0.0010
Epoch 10/60
1

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


26/26 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - loss: 0.2697 - val_loss: 0.1189 - learning_rate: 0.0010
Epoch 2/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0954 - val_loss: 0.1020 - learning_rate: 0.0010
Epoch 3/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0684 - val_loss: 0.1011 - learning_rate: 0.0010
Epoch 4/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0702 - val_loss: 0.0854 - learning_rate: 0.0010
Epoch 5/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0611 - val_loss: 0.1564 - learning_rate: 0.0010
Epoch 6/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0547 - val_loss: 0.0768 - learning_rate: 0.0010
Epoch 7/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0557 - val_loss: 0.1519 - learning_rate: 0.0010
Epoch 8/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0518 - val_loss: 0.0376 - learning_rate: 0.0010
Epoch 9/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0459 - val_loss: 0.0474 - learning_rate: 0.0010
Epoch 10/60
26/26 ━━━━━━━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - loss: 0.2216 - val_loss: 0.1144 - learning_rate: 0.0010
Epoch 2/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.1003 - val_loss: 0.0966 - learning_rate: 0.0010
Epoch 3/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0772 - val_loss: 0.0334 - learning_rate: 0.0010
Epoch 4/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0671 - val_loss: 0.0517 - learning_rate: 0.0010
Epoch 5/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0563 - val_loss: 0.0254 - learning_rate: 0.0010
Epoch 6/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0610 - val_loss: 0.0242 - learning_rate: 0.0010
Epoch 7/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0557 - val_loss: 0.1115 - learning_rate: 0.0010
Epoch 8/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0450 - val_loss: 0.0205 - learning_rate: 0.0010
Epoch 9/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0398 - val_loss: 0.0257 - learning_rate: 0.0010
Epoch 10/60
50/50 ━━━━━━━━━━━━━

{'DF1': {'High': {'MAE': 0.9886644263016546,
   'RMSE': 1.2293993486645873,
   'R2': 0.9463330197727204,
   'MAPE': 1.238253616868879},
  'Low': {'MAE': 1.0580835844341077,
   'RMSE': 1.296694508209455,
   'R2': 0.9448165371456173,
   'MAPE': 1.3554398944572619}},
 'DF2': {'High': {'MAE': 0.7445459926829617,
   'RMSE': 1.0059679120190161,
   'R2': 0.9735661675245537,
   'MAPE': 1.2765707899280343},
  'Low': {'MAE': 0.8421985738417683,
   'RMSE': 1.1286038107945464,
   'R2': 0.968626856394367,
   'MAPE': 1.502075181021422}},
 'DF3': {'High': {'MAE': 1.8923733359888981,
   'RMSE': 2.571714665238747,
   'R2': 0.7292322380606551,
   'MAPE': 2.48766911023215},
  'Low': {'MAE': 2.5191720661364103,
   'RMSE': 3.4190554311528305,
   'R2': 0.587603145456304,
   'MAPE': 3.437008376703636}},
 'DF4': {'High': {'MAE': 0.9942283891651729,
   'RMSE': 1.5464779089900316,
   'R2': 0.8323745661034618,
   'MAPE': 1.4995889606192616},
  'Low': {'MAE': 0.8246855278537695,
   'RMSE': 1.155822940338986,
   '

In [27]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from tcn import TCN


# ---------------- SEQUENCE CREATION ----------------
def create_seq(X, y, seq_len=25):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)


# ---------------- METRICS ----------------
def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true,  y_pred)
    mape = np.mean(np.abs((y_true - y_pred)/y_true))*100

    print(f"\n===== {label} =====")
    print("MAE :",  round(mae,3))
    print("RMSE:",  round(rmse,3))
    print("R2  :",  round(r2,3))
    print("MAPE:", round(mape,2), "%")

    return mae, rmse, r2, mape


# ---------------- BUILD TCN MODEL ----------------
def build_tcn(input_shape):
    model = Sequential([

        TCN(nb_filters=64,
            kernel_size=3,
            dilations=[1, 2, 4, 8, 16],
            dropout_rate=0.25,
            return_sequences=False,
            activation='relu',
            input_shape=input_shape),

        Dense(32, activation='relu'),
        Dropout(0.25),
        Dense(2)
    ])

    model.compile(optimizer='adam', loss='mse')
    return model


# ---------------- MAIN TCN FUNCTION ----------------
def run_TCN(df, name):

    print(f"\n================ TCN for {name} ================")

    # Features and target
    X = df[['Open','High','Low','Close']].values
    y = df[['High','Low']].values

    seq_len = 25
    train_size = max(int(len(df)*0.85), seq_len+3)
    train_size = min(train_size, len(df)-2)

    X_train_raw = X[:train_size]
    X_test_raw  = X[train_size:]

    y_train_raw = y[:train_size]
    y_test_raw  = y[train_size:]

    # Scaling
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train_raw)
    X_test_scaled  = scaler_X.transform(X_test_raw)

    y_train_scaled = scaler_y.fit_transform(y_train_raw)
    y_test_scaled  = scaler_y.transform(y_test_raw)

    X_scaled = np.vstack([X_train_scaled, X_test_scaled])
    y_scaled = np.vstack([y_train_scaled, y_test_scaled])

    # Sequence creation
    X_seq, y_seq = create_seq(X_scaled, y_scaled, seq_len)

    split = train_size - seq_len

    X_train, X_test = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]

    batch = 16 if len(X_train)>50 else 4

    model = build_tcn((seq_len, X_train.shape[2]))

    es = EarlyStopping(patience=6, restore_best_weights=True)
    lr = ReduceLROnPlateau(factor=0.5, patience=3)

    model.fit(
        X_train, y_train,
        epochs=60,
        batch_size=batch,
        validation_data=(X_test, y_test) if len(X_test)>1 else None,
        callbacks=[es, lr],
        verbose=1
    )

    # Prediction
    pred_scaled = model.predict(X_test)
    pred = scaler_y.inverse_transform(pred_scaled)
    y_true = scaler_y.inverse_transform(y_test)

    pred_high, pred_low = pred[:,0], pred[:,1]
    true_high, true_low = y_true[:,0], y_true[:,1]

    m1 = metrics(true_high, pred_high, f"TCN {name} High")
    m2 = metrics(true_low,  pred_low,  f"TCN {name} Low")

    return (m1, m2)


In [28]:
results_tcn = {}

for df, name in zip([df1, df2, df3, df4], ["DF1", "DF2", "DF3", "DF4"]):
    print(f"\n\n----- Training TCN for {name} -----")
    try:
        res = run_TCN(df, name)
        results_tcn[name] = {
            "High": {
                "MAE":  res[0][0],
                "RMSE": res[0][1],
                "R2":   res[0][2],
                "MAPE": res[0][3]
            },
            "Low": {
                "MAE":  res[1][0],
                "RMSE": res[1][1],
                "R2":   res[1][2],
                "MAPE": res[1][3]
            }
        }
    except Exception as e:
        print(f"Error in {name}: {e}")

results_tcn




----- Training TCN for DF1 -----

================ TCN for DF1 ================


d:\Anacondaa\envs\tfenv\lib\site-packages\tcn\tcn.py:268: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super(TCN, self).__init__(**kwargs)



Epoch 1/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - loss: 32.7875 - val_loss: 0.0659 - learning_rate: 0.0010
Epoch 2/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 2.6635 - val_loss: 0.0342 - learning_rate: 0.0010
Epoch 3/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 1.5538 - val_loss: 0.0329 - learning_rate: 0.0010
Epoch 4/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 1.1596 - val_loss: 0.0309 - learning_rate: 0.0010
Epoch 5/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.8706 - val_loss: 0.0301 - learning_rate: 0.0010
Epoch 6/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.7111 - val_loss: 0.0270 - learning_rate: 0.0010
Epoch 7/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.6702 - val_loss: 0.0275 - learning_rate: 0.0010
Epoch 8/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.6094 - val_loss: 0.0248 - learning_rate: 0.0010
Epoch 9/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.4590 - val_loss: 0.0234 - learning_rate: 0.0010
Epoch 10/60
52/52 ━

d:\Anacondaa\envs\tfenv\lib\site-packages\tcn\tcn.py:268: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super(TCN, self).__init__(**kwargs)


119/119 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 5.6885 - val_loss: 0.1645 - learning_rate: 0.0010
Epoch 2/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.4839 - val_loss: 0.0975 - learning_rate: 0.0010
Epoch 3/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.3481 - val_loss: 0.0788 - learning_rate: 0.0010
Epoch 4/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.2704 - val_loss: 0.0543 - learning_rate: 0.0010
Epoch 5/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.2231 - val_loss: 0.0360 - learning_rate: 0.0010
Epoch 6/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.1969 - val_loss: 0.0619 - learning_rate: 0.0010
Epoch 7/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.1847 - val_loss: 0.0537 - learning_rate: 0.0010
Epoch 8/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.1585 - val_loss: 0.0385 - learning_rate: 0.0010
Epoch 9/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.1444 - val_loss: 0.0338 - learning_rate: 5.0000e-04
Epoch 10/60
119/

d:\Anacondaa\envs\tfenv\lib\site-packages\tcn\tcn.py:268: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super(TCN, self).__init__(**kwargs)


26/26 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - loss: 17.7351 - val_loss: 0.3489 - learning_rate: 0.0010
Epoch 2/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 2.3359 - val_loss: 1.0794 - learning_rate: 0.0010
Epoch 3/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 1.0817 - val_loss: 0.8117 - learning_rate: 0.0010
Epoch 4/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.8490 - val_loss: 0.4559 - learning_rate: 0.0010
Epoch 5/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.8259 - val_loss: 0.5980 - learning_rate: 5.0000e-04
Epoch 6/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.6933 - val_loss: 0.6507 - learning_rate: 5.0000e-04
Epoch 7/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.7082 - val_loss: 0.6377 - learning_rate: 5.0000e-04
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step

===== TCN DF3 High =====
MAE : 9.878
RMSE: 10.894
R2  : -3.859
MAPE: 12.51 %

===== TCN DF3 Low =====
MAE : 5.781
RMSE: 6.583
R2  : -0.529
MAPE: 7.82 %


----- Training TCN for DF4 -----

=======

d:\Anacondaa\envs\tfenv\lib\site-packages\tcn\tcn.py:268: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super(TCN, self).__init__(**kwargs)


50/50 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - loss: 16.2335 - val_loss: 1.5916 - learning_rate: 0.0010
Epoch 2/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.5030 - val_loss: 0.6400 - learning_rate: 0.0010
Epoch 3/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.0179 - val_loss: 0.6373 - learning_rate: 0.0010
Epoch 4/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.7514 - val_loss: 0.4104 - learning_rate: 0.0010
Epoch 5/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.6847 - val_loss: 0.3495 - learning_rate: 0.0010
Epoch 6/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.6337 - val_loss: 0.1729 - learning_rate: 0.0010
Epoch 7/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.5204 - val_loss: 0.2655 - learning_rate: 0.0010
Epoch 8/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.5475 - val_loss: 0.1963 - learning_rate: 0.0010
Epoch 9/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.4967 - val_loss: 0.1717 - learning_rate: 0.0010
Epoch 10/60
50/50 ━━━━━━━━━━━━━━━━━━━

{'DF1': {'High': {'MAE': 1.7198804554186369,
   'RMSE': 2.083677210412063,
   'R2': 0.8458363142263668,
   'MAPE': 2.1170753768852624},
  'Low': {'MAE': 2.085456496790836,
   'RMSE': 2.5852570650795097,
   'R2': 0.7806482649478539,
   'MAPE': 2.5811167152476844}},
 'DF2': {'High': {'MAE': 1.604737876443302,
   'RMSE': 1.9878826824334392,
   'R2': 0.8967777365401757,
   'MAPE': 2.7979464530333913},
  'Low': {'MAE': 2.109506909987506,
   'RMSE': 2.576400585164204,
   'R2': 0.8365060106699975,
   'MAPE': 3.8574625091883585}},
 'DF3': {'High': {'MAE': 9.87798419751619,
   'RMSE': 10.89378204931149,
   'R2': -3.858573271657675,
   'MAPE': 12.508548471622365},
  'Low': {'MAE': 5.781279614097194,
   'RMSE': 6.582859464439165,
   'R2': -0.5287350664931714,
   'MAPE': 7.8249529868025425}},
 'DF4': {'High': {'MAE': 2.4530357987913365,
   'RMSE': 2.8712243535158426,
   'R2': 0.42218776746990705,
   'MAPE': 3.807851796625504},
  'Low': {'MAE': 1.3318189333562984,
   'RMSE': 1.7546060904744887,
   

In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau



# ---------------- SEQUENCE CREATION ----------------
def create_seq(X, y, seq_len=25):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)



# ---------------- METRICS ----------------
def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true,  y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"\n===== {label} =====")
    print("MAE :",  round(mae, 3))
    print("RMSE:",  round(rmse, 3))
    print("R2  :",  round(r2, 3))
    print("MAPE:", round(mape, 2), "%")

    return mae, rmse, r2, mape



# ---------------- BUILD LSTM MODEL ----------------
def build_lstm(input_shape):

    model = Sequential([

        LSTM(96, return_sequences=True, input_shape=input_shape),
        Dropout(0.25),

        LSTM(64, return_sequences=False),
        Dropout(0.25),

        Dense(32, activation='relu'),
        Dense(2)
    ])

    model.compile(optimizer='adam', loss='mse')
    return model



# ---------------- MAIN FUNCTION ----------------
def run_LSTM(df, name):

    print(f"\n================ LSTM for {name} ================")

    # Features and target
    X = df[['Open', 'High', 'Low', 'Close']].values
    y = df[['High', 'Low']].values

    seq_len = 25
    train_size = max(int(len(df) * 0.85), seq_len + 3)
    train_size = min(train_size, len(df) - 2)

    X_train_raw = X[:train_size]
    X_test_raw  = X[train_size:]

    y_train_raw = y[:train_size]
    y_test_raw  = y[train_size:]

    # Scaling
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train_raw)
    X_test_scaled  = scaler_X.transform(X_test_raw)

    y_train_scaled = scaler_y.fit_transform(y_train_raw)
    y_test_scaled  = scaler_y.transform(y_test_raw)

    X_scaled = np.vstack([X_train_scaled, X_test_scaled])
    y_scaled = np.vstack([y_train_scaled, y_test_scaled])

    # Create sequences
    X_seq, y_seq = create_seq(X_scaled, y_scaled, seq_len)

    split = train_size - seq_len
    X_train, X_test = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]

    # Adaptive batch size
    batch_size = 16 if len(X_train) > 50 else 4

    # Build model
    model = build_lstm((seq_len, X_train.shape[2]))

    # Callbacks
    es = EarlyStopping(patience=6, restore_best_weights=True)
    lr = ReduceLROnPlateau(factor=0.5, patience=3)

    # Train
    model.fit(
        X_train, y_train,
        epochs=60,
        batch_size=batch_size,
        validation_data=(X_test, y_test) if len(X_test) > 1 else None,
        callbacks=[es, lr],
        verbose=1
    )

    # Prediction
    pred_scaled = model.predict(X_test)
    pred = scaler_y.inverse_transform(pred_scaled)
    y_true = scaler_y.inverse_transform(y_test)

    pred_high, pred_low = pred[:, 0], pred[:, 1]
    true_high, true_low = y_true[:, 0], y_true[:, 1]

    m1 = metrics(true_high, pred_high, f"LSTM {name} High")
    m2 = metrics(true_low,  pred_low,  f"LSTM {name} Low")

    return (m1, m2)



# ================= RUN LSTM ON ALL DATASETS ================= #

print("\n\n====== RUNNING LSTM ON ALL DATASETS ======\n")

results_lstm = {}

# df1, df2, df3, df4 must already be defined
for df, name in zip([df1, df2, df3, df4], ["DF1", "DF2", "DF3", "DF4"]):
    print(f"\n\n----- Training LSTM for {name} -----")
    try:
        res = run_LSTM(df, name)
        results_lstm[name] = {
            "High": {
                "MAE":  res[0][0],
                "RMSE": res[0][1],
                "R2":   res[0][2],
                "MAPE": res[0][3]
            },
            "Low": {
                "MAE":  res[1][0],
                "RMSE": res[1][1],
                "R2":   res[1][2],
                "MAPE": res[1][3]
            }
        }
    except Exception as e:
        print(f"Error in {name}: {e}")

print("\n\n============= FINAL LSTM RESULTS =============\n")
results_lstm




====== RUNNING LSTM ON ALL DATASETS ======



----- Training LSTM for DF1 -----

================ LSTM for DF1 ================


d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 0.1673 - val_loss: 0.0143 - learning_rate: 0.0010
Epoch 2/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0485 - val_loss: 0.0116 - learning_rate: 0.0010
Epoch 3/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0467 - val_loss: 0.0151 - learning_rate: 0.0010
Epoch 4/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0389 - val_loss: 0.0102 - learning_rate: 0.0010
Epoch 5/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0307 - val_loss: 0.0102 - learning_rate: 0.0010
Epoch 6/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0316 - val_loss: 0.0088 - learning_rate: 0.0010
Epoch 7/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0324 - val_loss: 0.0087 - learning_rate: 0.0010
Epoch 8/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0311 - val_loss: 0.0077 - learning_rate: 0.0010
Epoch 9/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0261 - val_loss: 0.0065 - learning_rate: 0.0010
Epoch 10/60
52/52 ━━━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


119/119 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - loss: 0.0834 - val_loss: 0.0103 - learning_rate: 0.0010
Epoch 2/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0260 - val_loss: 0.0088 - learning_rate: 0.0010
Epoch 3/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0201 - val_loss: 0.0100 - learning_rate: 0.0010
Epoch 4/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0196 - val_loss: 0.0131 - learning_rate: 0.0010
Epoch 5/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0190 - val_loss: 0.0056 - learning_rate: 0.0010
Epoch 6/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0155 - val_loss: 0.0050 - learning_rate: 0.0010
Epoch 7/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0145 - val_loss: 0.0055 - learning_rate: 0.0010
Epoch 8/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0135 - val_loss: 0.0056 - learning_rate: 0.0010
Epoch 9/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0127 - val_loss: 0.0071 - learning_rate: 0.0010
Epoch 10/60
119/11

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - loss: 0.3668 - val_loss: 0.1349 - learning_rate: 0.0010
Epoch 2/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0952 - val_loss: 0.0776 - learning_rate: 0.0010
Epoch 3/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0828 - val_loss: 0.1321 - learning_rate: 0.0010
Epoch 4/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0755 - val_loss: 0.0818 - learning_rate: 0.0010
Epoch 5/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0646 - val_loss: 0.0739 - learning_rate: 0.0010
Epoch 6/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0670 - val_loss: 0.1032 - learning_rate: 0.0010
Epoch 7/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0643 - val_loss: 0.0631 - learning_rate: 0.0010
Epoch 8/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0604 - val_loss: 0.1308 - learning_rate: 0.0010
Epoch 9/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0662 - val_loss: 0.1831 - learning_rate: 0.0010
Epoch 10/60
26/26 ━━━━━━━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - loss: 0.2986 - val_loss: 0.1627 - learning_rate: 0.0010
Epoch 2/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.1199 - val_loss: 0.1351 - learning_rate: 0.0010
Epoch 3/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.1081 - val_loss: 0.1711 - learning_rate: 0.0010
Epoch 4/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0951 - val_loss: 0.0352 - learning_rate: 0.0010
Epoch 5/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0789 - val_loss: 0.0547 - learning_rate: 0.0010
Epoch 6/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0758 - val_loss: 0.0339 - learning_rate: 0.0010
Epoch 7/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0657 - val_loss: 0.0284 - learning_rate: 0.0010
Epoch 8/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0552 - val_loss: 0.0227 - learning_rate: 0.0010
Epoch 9/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0613 - val_loss: 0.0214 - learning_rate: 0.0010
Epoch 10/60
50/50 ━━━━━━━━━━━━

{'DF1': {'High': {'MAE': 1.0430764148109837,
   'RMSE': 1.3082020342923357,
   'R2': 0.9392325738792185,
   'MAPE': 1.3058917145633981},
  'Low': {'MAE': 1.1643218994140625,
   'RMSE': 1.409520162765833,
   'R2': 0.9347957067301154,
   'MAPE': 1.4917732295766717}},
 'DF2': {'High': {'MAE': 0.8487095215741326,
   'RMSE': 1.1346872102148382,
   'R2': 0.9663686580541369,
   'MAPE': 1.452019914273785},
  'Low': {'MAE': 1.0381898992201861,
   'RMSE': 1.3269009667268041,
   'R2': 0.9566337335661949,
   'MAPE': 1.8390660953983908}},
 'DF3': {'High': {'MAE': 3.0787972901996814,
   'RMSE': 3.628535080472225,
   'R2': 0.46096862481765977,
   'MAPE': 3.901918063017716},
  'Low': {'MAE': 3.4003169411107113,
   'RMSE': 4.153769403102958,
   'R2': 0.3913216895379549,
   'MAPE': 4.529380700794588}},
 'DF4': {'High': {'MAE': 0.9717489268681775,
   'RMSE': 1.4748028110194225,
   'R2': 0.8475524717374687,
   'MAPE': 1.4638150235936658},
  'Low': {'MAE': 0.8935027971659621,
   'RMSE': 1.245618004985985,


In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import ConvLSTM2D, Dropout, Flatten, Dense
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau



# ---------------- SEQUENCE CREATION ----------------
def create_seq(X, y, seq_len=25):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)



# ---------------- METRICS ----------------
def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true,  y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"\n===== {label} =====")
    print("MAE :",  round(mae, 3))
    print("RMSE:",  round(rmse, 3))
    print("R2  :",  round(r2, 3))
    print("MAPE:", round(mape, 2), "%")

    return mae, rmse, r2, mape



# ---------------- BUILD ConvLSTM MODEL ----------------
def build_convlstm(input_shape):

    model = Sequential([

        ConvLSTM2D(
            filters=64,
            kernel_size=(1, 2),
            return_sequences=True,
            activation='tanh',
            padding='same',
            input_shape=input_shape
        ),
        Dropout(0.25),

        ConvLSTM2D(
            filters=32,
            kernel_size=(1, 2),
            return_sequences=False,
            activation='tanh',
            padding='same'
        ),
        Dropout(0.25),

        Flatten(),

        Dense(64, activation='relu'),
        Dense(2)
    ])

    model.compile(optimizer='adam', loss='mse')
    return model



# ---------------- MAIN FUNCTION ----------------
def run_ConvLSTM(df, name):

    print(f"\n================ ConvLSTM for {name} ================")

    # Features and target
    X = df[['Open', 'High', 'Low', 'Close']].values
    y = df[['High', 'Low']].values

    seq_len = 25
    train_size = max(int(len(df) * 0.85), seq_len + 3)
    train_size = min(train_size, len(df) - 2)

    X_train_raw = X[:train_size]
    X_test_raw  = X[train_size:]

    y_train_raw = y[:train_size]
    y_test_raw  = y[train_size:]

    # Scaling
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train_raw)
    X_test_scaled  = scaler_X.transform(X_test_raw)

    y_train_scaled = scaler_y.fit_transform(y_train_raw)
    y_test_scaled  = scaler_y.transform(y_test_raw)

    # Merge for sequence creation
    X_scaled = np.vstack([X_train_scaled, X_test_scaled])
    y_scaled = np.vstack([y_train_scaled, y_test_scaled])

    # Create sequences
    X_seq, y_seq = create_seq(X_scaled, y_scaled, seq_len)

    # Split into train/test
    split = train_size - seq_len
    X_train_seq, X_test_seq = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]

    # ConvLSTM requires 5D input: (batch, seq_len, spatial_x, spatial_y, channels)
    X_train = X_train_seq.reshape(-1, seq_len, 1, X_train_raw.shape[1], 1)
    X_test  = X_test_seq.reshape(-1, seq_len, 1, X_train_raw.shape[1], 1)

    # Adaptive batch size
    batch_size = 8 if len(X_train) > 50 else 2

    # Build model
    model = build_convlstm((seq_len, 1, X_train_raw.shape[1], 1))

    # Callbacks
    es = EarlyStopping(patience=6, restore_best_weights=True)
    lr = ReduceLROnPlateau(factor=0.5, patience=3)

    # Train model
    model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=batch_size,
        validation_data=(X_test, y_test) if len(X_test) > 1 else None,
        callbacks=[es, lr],
        verbose=1
    )

    # Predict
    pred_scaled = model.predict(X_test)
    pred = scaler_y.inverse_transform(pred_scaled)
    y_true = scaler_y.inverse_transform(y_test)

    pred_high, pred_low = pred[:, 0], pred[:, 1]
    true_high, true_low = y_true[:, 0], y_true[:, 1]

    m1 = metrics(true_high, pred_high, f"ConvLSTM {name} High")
    m2 = metrics(true_low,  pred_low,  f"ConvLSTM {name} Low")

    return (m1, m2)



# ================= RUN ConvLSTM ON ALL DATASETS ================= #
print("\n\n====== RUNNING ConvLSTM ON ALL DATASETS ======\n")

results_convlstm = {}

for df_regime, name in zip([df1, df2, df3, df4], ["DF1", "DF2", "DF3", "DF4"]):
    print(f"\n\n----- Training ConvLSTM for {name} -----")
    try:
        res = run_ConvLSTM(df_regime, name)
        results_convlstm[name] = {
            "High": {
                "MAE":  res[0][0],
                "RMSE": res[0][1],
                "R2":   res[0][2],
                "MAPE": res[0][3]
            },
            "Low": {
                "MAE":  res[1][0],
                "RMSE": res[1][1],
                "R2":   res[1][2],
                "MAPE": res[1][3]
            }
        }
    except Exception as e:
        print(f"Error in {name}: {e}")

print("\n\n============= FINAL ConvLSTM RESULTS =============\n")
results_convlstm




====== RUNNING ConvLSTM ON ALL DATASETS ======



----- Training ConvLSTM for DF1 -----

================ ConvLSTM for DF1 ================
Epoch 1/50


d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


104/104 ━━━━━━━━━━━━━━━━━━━━ 6s 26ms/step - loss: 0.1024 - val_loss: 0.0118 - learning_rate: 0.0010
Epoch 2/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - loss: 0.0359 - val_loss: 0.0101 - learning_rate: 0.0010
Epoch 3/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 0.0306 - val_loss: 0.0074 - learning_rate: 0.0010
Epoch 4/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 0.0338 - val_loss: 0.0088 - learning_rate: 0.0010
Epoch 5/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - loss: 0.0245 - val_loss: 0.0101 - learning_rate: 0.0010
Epoch 6/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - loss: 0.0217 - val_loss: 0.0064 - learning_rate: 0.0010
Epoch 7/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 0.0217 - val_loss: 0.0050 - learning_rate: 0.0010
Epoch 8/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 0.0200 - val_loss: 0.0055 - learning_rate: 0.0010
Epoch 9/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 0.0192 - val_loss: 0.0051 - learning_rate: 0.0010
Epoch 10/50


d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


238/238 ━━━━━━━━━━━━━━━━━━━━ 8s 24ms/step - loss: 0.0554 - val_loss: 0.0179 - learning_rate: 0.0010
Epoch 2/50
238/238 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 0.0184 - val_loss: 0.0071 - learning_rate: 0.0010
Epoch 3/50
238/238 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 0.0147 - val_loss: 0.0049 - learning_rate: 0.0010
Epoch 4/50
238/238 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 0.0139 - val_loss: 0.0048 - learning_rate: 0.0010
Epoch 5/50
238/238 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 0.0115 - val_loss: 0.0145 - learning_rate: 0.0010
Epoch 6/50
238/238 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - loss: 0.0106 - val_loss: 0.0039 - learning_rate: 0.0010
Epoch 7/50
238/238 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 0.0100 - val_loss: 0.0125 - learning_rate: 0.0010
Epoch 8/50
238/238 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - loss: 0.0096 - val_loss: 0.0031 - learning_rate: 0.0010
Epoch 9/50
238/238 ━━━━━━━━━━━━━━━━━━━━ 6s 24ms/step - loss: 0.0094 - val_loss: 0.0029 - learning_rate: 0.0010
Epoch 10/50


d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


51/51 ━━━━━━━━━━━━━━━━━━━━ 5s 41ms/step - loss: 0.1916 - val_loss: 0.1518 - learning_rate: 0.0010
Epoch 2/50
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.0915 - val_loss: 0.1194 - learning_rate: 0.0010
Epoch 3/50
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.0585 - val_loss: 0.1189 - learning_rate: 0.0010
Epoch 4/50
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.0561 - val_loss: 0.0566 - learning_rate: 0.0010
Epoch 5/50
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.0604 - val_loss: 0.1378 - learning_rate: 0.0010
Epoch 6/50
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.0510 - val_loss: 0.0748 - learning_rate: 0.0010
Epoch 7/50
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.0523 - val_loss: 0.0456 - learning_rate: 0.0010
Epoch 8/50
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.0592 - val_loss: 0.0403 - learning_rate: 0.0010
Epoch 9/50
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.0555 - val_loss: 0.1121 - learning_rate: 0.0010
Epoch 10/50
51/51 ━━━━━━━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 28ms/step - loss: 0.2032 - val_loss: 0.0711 - learning_rate: 0.0010
Epoch 2/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 0.0997 - val_loss: 0.0507 - learning_rate: 0.0010
Epoch 3/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 0.0808 - val_loss: 0.0483 - learning_rate: 0.0010
Epoch 4/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 0.0666 - val_loss: 0.0252 - learning_rate: 0.0010
Epoch 5/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 0.0540 - val_loss: 0.0208 - learning_rate: 0.0010
Epoch 6/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 0.0538 - val_loss: 0.0260 - learning_rate: 0.0010
Epoch 7/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 0.0395 - val_loss: 0.0388 - learning_rate: 0.0010
Epoch 8/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 0.0429 - val_loss: 0.0189 - learning_rate: 0.0010
Epoch 9/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 0.0387 - val_loss: 0.0149 - learning_rate: 0.0010
Epoch 10/50


{'DF1': {'High': {'MAE': 0.9703703428569593,
   'RMSE': 1.193833359781852,
   'R2': 0.9493932296473645,
   'MAPE': 1.2115721400281736},
  'Low': {'MAE': 1.011248739142167,
   'RMSE': 1.2447571514986107,
   'R2': 0.9491486055253454,
   'MAPE': 1.298548435379449}},
 'DF2': {'High': {'MAE': 0.6565568250768323,
   'RMSE': 0.9198060966732494,
   'R2': 0.9779003983728981,
   'MAPE': 1.1211986220292367},
  'Low': {'MAE': 0.8269280040965359,
   'RMSE': 1.0999248247656934,
   'R2': 0.9702010459406156,
   'MAPE': 1.4651968107865754}},
 'DF3': {'High': {'MAE': 1.6191100070351048,
   'RMSE': 2.0194055823073134,
   'R2': 0.8330453585241366,
   'MAPE': 2.1115680793671396},
  'Low': {'MAE': 2.049639952810187,
   'RMSE': 2.658517524710265,
   'R2': 0.7506656491383075,
   'MAPE': 2.8108877732517024}},
 'DF4': {'High': {'MAE': 0.8406478672811429,
   'RMSE': 1.2402356491046416,
   'R2': 0.8921895410707092,
   'MAPE': 1.2754134491391775},
  'Low': {'MAE': 0.6963404564008322,
   'RMSE': 1.0582850941803783,

In [23]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from catboost import CatBoostRegressor


# ---------------- SEQUENCE CREATION ----------------
def create_seq(X, y, seq_len=25):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i].flatten())  # Flatten for CatBoost
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)


# ---------------- METRICS ----------------
def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true,  y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"\n===== {label} =====")
    print("MAE :",  round(mae, 3))
    print("RMSE:",  round(rmse, 3))
    print("R2  :",  round(r2, 3))
    print("MAPE:", round(mape, 2), "%")

    return mae, rmse, r2, mape


# ---------------- BUILD & RUN CATBOOST MODEL ----------------
def run_CatBoost(df, name):

    print(f"\n================ CatBoost for {name} ================")

    # Features and target
    X = df[['Open', 'High', 'Low', 'Close']].values
    y = df[['High', 'Low']].values

    seq_len = 25
    train_size = max(int(len(df) * 0.85), seq_len + 3)
    train_size = min(train_size, len(df) - 2)

    X_train_raw = X[:train_size]
    X_test_raw  = X[train_size:]

    y_train_raw = y[:train_size]
    y_test_raw  = y[train_size:]

    # Scaling
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train_raw)
    X_test_scaled  = scaler_X.transform(X_test_raw)

    y_train_scaled = scaler_y.fit_transform(y_train_raw)
    y_test_scaled  = scaler_y.transform(y_test_raw)

    # Merge for sequence creation
    X_scaled = np.vstack([X_train_scaled, X_test_scaled])
    y_scaled = np.vstack([y_train_scaled, y_test_scaled])

    # Create sequences
    X_seq, y_seq = create_seq(X_scaled, y_scaled, seq_len)

    # Split into train/test
    split = train_size - seq_len
    X_train, X_test = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]

    # ---------------- CatBoost Model ----------------
    model = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.05,
        depth=8,
        loss_function='MultiRMSE',
        verbose=False
    )

    # Train
    model.fit(X_train, y_train)

    # Predict
    pred_scaled = model.predict(X_test)
    pred = scaler_y.inverse_transform(pred_scaled)
    y_true = scaler_y.inverse_transform(y_test)

    pred_high, pred_low = pred[:, 0], pred[:, 1]
    true_high, true_low = y_true[:, 0], y_true[:, 1]

    m1 = metrics(true_high, pred_high, f"CatBoost {name} High")
    m2 = metrics(true_low,  pred_low,  f"CatBoost {name} Low")

    return (m1, m2)



# ================= RUN CATBOOST ON ALL DATASETS ================= #

print("\n\n====== RUNNING CatBoost ON ALL DATASETS ======\n")

results_catboost = {}

# df1, df2, df3, df4 must already be created before this
for df_regime, name in zip([df1, df2, df3, df4], ["DF1", "DF2", "DF3", "DF4"]):
    print(f"\n\n----- Training CatBoost for {name} -----")
    try:
        res = run_CatBoost(df_regime, name)
        results_catboost[name] = {
            "High": {
                "MAE":  res[0][0],
                "RMSE": res[0][1],
                "R2":   res[0][2],
                "MAPE": res[0][3]
            },
            "Low": {
                "MAE":  res[1][0],
                "RMSE": res[1][1],
                "R2":   res[1][2],
                "MAPE": res[1][3]
            }
        }
    except Exception as e:
        print(f"Error in {name}: {e}")

print("\n\n============= FINAL CatBoost RESULTS =============\n")
results_catboost




====== RUNNING CatBoost ON ALL DATASETS ======



----- Training CatBoost for DF1 -----

================ CatBoost for DF1 ================

===== CatBoost DF1 High =====
MAE : 1.108
RMSE: 1.39
R2  : 0.931
MAPE: 1.36 %

===== CatBoost DF1 Low =====
MAE : 1.054
RMSE: 1.319
R2  : 0.943
MAPE: 1.34 %


----- Training CatBoost for DF2 -----

================ CatBoost for DF2 ================

===== CatBoost DF2 High =====
MAE : 1.097
RMSE: 1.43
R2  : 0.947
MAPE: 1.86 %

===== CatBoost DF2 Low =====
MAE : 1.13
RMSE: 1.483
R2  : 0.946
MAPE: 1.99 %


----- Training CatBoost for DF3 -----

================ CatBoost for DF3 ================

===== CatBoost DF3 High =====
MAE : 5.279
RMSE: 6.583
R2  : -0.774
MAPE: 6.52 %

===== CatBoost DF3 Low =====
MAE : 5.651
RMSE: 7.109
R2  : -0.783
MAPE: 7.2 %


----- Training CatBoost for DF4 -----

================ CatBoost for DF4 ================

===== CatBoost DF4 High =====
MAE : 2.105
RMSE: 2.943
R2  : 0.393
MAPE: 3.26 %

===== CatBoost DF4 Low ===

{'DF1': {'High': {'MAE': 1.1084922726827653,
   'RMSE': 1.3899052281105244,
   'R2': 0.9314051393102333,
   'MAPE': 1.3627178110799265},
  'Low': {'MAE': 1.0542414599485896,
   'RMSE': 1.3193846652505126,
   'R2': 0.9428683889446073,
   'MAPE': 1.3367853152169775}},
 'DF2': {'High': {'MAE': 1.0974504505418183,
   'RMSE': 1.4302029909052032,
   'R2': 0.946569745699045,
   'MAPE': 1.8598645828551867},
  'Low': {'MAE': 1.1298981437597035,
   'RMSE': 1.4830147024395501,
   'R2': 0.9458291156113837,
   'MAPE': 1.9911700425812529}},
 'DF3': {'High': {'MAE': 5.279183328236652,
   'RMSE': 6.58331484426252,
   'R2': -0.7743555262313011,
   'MAPE': 6.518210520330362},
  'Low': {'MAE': 5.6510465342996214,
   'RMSE': 7.108808523945237,
   'R2': -0.7827756506283039,
   'MAPE': 7.1995984210157085}},
 'DF4': {'High': {'MAE': 2.1048159364224026,
   'RMSE': 2.942600871739888,
   'R2': 0.3931027181354221,
   'MAPE': 3.26083219434233},
  'Low': {'MAE': 3.093743986845251,
   'RMSE': 3.721186237335006,
   

In [24]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.multioutput import MultiOutputRegressor

from xgboost import XGBRegressor


# ---------------- SEQUENCE CREATION ----------------
def create_seq(X, y, seq_len=25):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i].flatten())  # flatten for XGBoost
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)


# ---------------- METRICS ----------------
def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true,  y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"\n===== {label} =====")
    print("MAE :",  round(mae, 3))
    print("RMSE:",  round(rmse, 3))
    print("R2  :",  round(r2, 3))
    print("MAPE:", round(mape, 2), "%")

    return mae, rmse, r2, mape


# ---------------- BUILD & RUN XGBOOST MODEL ----------------
def run_XGBoost(df, name):

    print(f"\n================ XGBoost for {name} ================")

    # Features and target
    X = df[['Open', 'High', 'Low', 'Close']].values
    y = df[['High', 'Low']].values   # Multi-output prediction

    seq_len = 25
    train_size = max(int(len(df) * 0.85), seq_len + 3)
    train_size = min(train_size, len(df) - 2)

    X_train_raw = X[:train_size]
    X_test_raw  = X[train_size:]

    y_train_raw = y[:train_size]
    y_test_raw  = y[train_size:]

    # Scaling
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train_raw)
    X_test_scaled  = scaler_X.transform(X_test_raw)

    y_train_scaled = scaler_y.fit_transform(y_train_raw)
    y_test_scaled  = scaler_y.transform(y_test_raw)

    X_scaled = np.vstack([X_train_scaled, X_test_scaled])
    y_scaled = np.vstack([y_train_scaled, y_test_scaled])

    # Sequence creation
    X_seq, y_seq = create_seq(X_scaled, y_scaled, seq_len)

    split = train_size - seq_len
    X_train, X_test = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]


    # ---------------- XGBoost Model ----------------
    model = MultiOutputRegressor(
        XGBRegressor(
            n_estimators=600,
            learning_rate=0.05,
            max_depth=8,
            subsample=0.9,
            colsample_bytree=0.9,
            objective='reg:squarederror',
            tree_method="hist",
            eval_metric="rmse"
        )
    )

    # Train
    model.fit(X_train, y_train)

    # Predict
    pred_scaled = model.predict(X_test)
    pred = scaler_y.inverse_transform(pred_scaled)
    y_true = scaler_y.inverse_transform(y_test)

    pred_high, pred_low = pred[:, 0], pred[:, 1]
    true_high, true_low = y_true[:, 0], y_true[:, 1]

    m1 = metrics(true_high, pred_high, f"XGBoost {name} High")
    m2 = metrics(true_low, pred_low,  f"XGBoost {name} Low")

    return (m1, m2)




# ================= RUN XGBOOST ON ALL DATASETS ================= #

print("\n\n====== RUNNING XGBoost ON ALL DATASETS ======\n")

results_xgboost = {}

for df_regime, name in zip([df1, df2, df3, df4], ["DF1", "DF2", "DF3", "DF4"]):
    print(f"\n\n----- Training XGBoost for {name} -----")
    try:
        res = run_XGBoost(df_regime, name)
        results_xgboost[name] = {
            "High": {
                "MAE":  res[0][0],
                "RMSE": res[0][1],
                "R2":   res[0][2],
                "MAPE": res[0][3]
            },
            "Low": {
                "MAE":  res[1][0],
                "RMSE": res[1][1],
                "R2":   res[1][2],
                "MAPE": res[1][3]
            }
        }
    except Exception as e:
        print(f"Error in {name}: {e}")

print("\n\n============= FINAL XGBoost RESULTS =============\n")
results_xgboost




====== RUNNING XGBoost ON ALL DATASETS ======



----- Training XGBoost for DF1 -----

================ XGBoost for DF1 ================

===== XGBoost DF1 High =====
MAE : 0.878
RMSE: 1.081
R2  : 0.958
MAPE: 1.08 %

===== XGBoost DF1 Low =====
MAE : 0.787
RMSE: 0.998
R2  : 0.967
MAPE: 1.01 %


----- Training XGBoost for DF2 -----

================ XGBoost for DF2 ================

===== XGBoost DF2 High =====
MAE : 0.64
RMSE: 0.911
R2  : 0.978
MAPE: 1.09 %

===== XGBoost DF2 Low =====
MAE : 0.801
RMSE: 1.08
R2  : 0.971
MAPE: 1.4 %


----- Training XGBoost for DF3 -----

================ XGBoost for DF3 ================

===== XGBoost DF3 High =====
MAE : 4.362
RMSE: 5.324
R2  : -0.16
MAPE: 5.42 %

===== XGBoost DF3 Low =====
MAE : 6.3
RMSE: 7.993
R2  : -1.254
MAPE: 8.03 %


----- Training XGBoost for DF4 -----

================ XGBoost for DF4 ================

===== XGBoost DF4 High =====
MAE : 2.437
RMSE: 3.255
R2  : 0.258
MAPE: 3.81 %

===== XGBoost DF4 Low =====
MAE : 4.684
RMSE

{'DF1': {'High': {'MAE': 0.8784337796662987,
   'RMSE': 1.081403600701105,
   'R2': 0.9584762254920149,
   'MAPE': 1.079034106637663},
  'Low': {'MAE': 0.7868733657033818,
   'RMSE': 0.997756117542104,
   'R2': 0.9673274971002583,
   'MAPE': 1.0087312914959619}},
 'DF2': {'High': {'MAE': 0.6400400161743164,
   'RMSE': 0.9112593667446217,
   'R2': 0.9783091841052112,
   'MAPE': 1.0865386796751333},
  'Low': {'MAE': 0.8007092644186581,
   'RMSE': 1.0798071314956108,
   'R2': 0.9712811268408915,
   'MAPE': 1.4044307712177575}},
 'DF3': {'High': {'MAE': 4.361819719013415,
   'RMSE': 5.323901126810855,
   'R2': -0.16040971232918722,
   'MAPE': 5.420864944107808},
  'Low': {'MAE': 6.299849309419331,
   'RMSE': 7.9928144669472125,
   'R2': -1.2537333152169445,
   'MAPE': 8.029543113405486}},
 'DF4': {'High': {'MAE': 2.437244911716409,
   'RMSE': 3.2545725444493896,
   'R2': 0.2575958572821361,
   'MAPE': 3.814114047880537},
  'Low': {'MAE': 4.684027083932537,
   'RMSE': 5.433159176960964,
   

In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# ---------------- SEQUENCE CREATION ----------------
def create_seq(X, y, seq_len=25):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)



# ---------------- METRICS ----------------
def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true,  y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"\n===== {label} =====")
    print("MAE :", round(mae, 3))
    print("RMSE:", round(rmse, 3))
    print("R2  :", round(r2, 3))
    print("MAPE:", round(mape, 2), "%")

    return mae, rmse, r2, mape



# ---------------- BUILD Conv1D + LSTM MODEL ----------------
def build_conv_lstm_model(seq_len, num_features):

    model = Sequential([
        Conv1D(filters=64, kernel_size=3, activation='relu', padding='same',
               input_shape=(seq_len, num_features)),
        Dropout(0.2),

        Conv1D(filters=32, kernel_size=3, activation='relu', padding='same'),
        Dropout(0.2),

        LSTM(64, return_sequences=False),
        Dropout(0.2),

        Dense(32, activation='relu'),
        Dense(2)   # Output: High, Low
    ])

    model.compile(optimizer='adam', loss='mse')
    return model



# ---------------- MAIN FUNCTION ----------------
def run_ConvLSTM(df, name):

    print(f"\n================ Conv1D-LSTM for {name} ================")

    # Features and target
    X = df[['Open', 'High', 'Low', 'Close']].values
    y = df[['High', 'Low']].values

    seq_len = 25
    train_size = max(int(len(df) * 0.85), seq_len + 3)
    train_size = min(train_size, len(df) - 2)

    # Scale data
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y)

    # Create sequences
    X_seq, y_seq = create_seq(X_scaled, y_scaled, seq_len)

    # Dataset split
    split = train_size - seq_len
    X_train, X_test = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]

    print("Training samples:", len(X_train), " | Testing samples:", len(X_test))

    # Build model
    model = build_conv_lstm_model(seq_len, X.shape[1])

    # Callbacks
    es = EarlyStopping(patience=8, restore_best_weights=True)
    lr = ReduceLROnPlateau(factor=0.5, patience=4)

    # Train model
    model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=16,
        validation_data=(X_test, y_test),
        callbacks=[es, lr],
        verbose=1
    )

    # Predict
    pred_scaled = model.predict(X_test)
    pred = scaler_y.inverse_transform(pred_scaled)
    y_true = scaler_y.inverse_transform(y_test)

    pred_high, pred_low = pred[:, 0], pred[:, 1]
    true_high, true_low = y_true[:, 0], y_true[:, 1]

    m1 = metrics(true_high, pred_high, f"ConvLSTM {name} High")
    m2 = metrics(true_low,  pred_low,  f"ConvLSTM {name} Low")

    return (m1, m2)



# ================= RUN ConvLSTM ON ALL DATASETS ================= #
print("\n\n====== RUNNING Conv1D-LSTM ON ALL DATASETS ======\n")

results_convlstm = {}

for df_regime, name in zip([df1, df2, df3, df4], ["DF1", "DF2", "DF3", "DF4"]):
    print(f"\n\n----- Training Conv1D-LSTM for {name} -----")
    try:
        res = run_ConvLSTM(df_regime, name)
        results_convlstm[name] = {
            "High": {
                "MAE":  res[0][0],
                "RMSE": res[0][1],
                "R2":   res[0][2],
                "MAPE": res[0][3]
            },
            "Low": {
                "MAE":  res[1][0],
                "RMSE": res[1][1],
                "R2":   res[1][2],
                "MAPE": res[1][3]
            }
        }
    except Exception as e:
        print(f"Error in {name}: {e}")

print("\n\n============= FINAL Conv1D-LSTM RESULTS =============\n")
results_convlstm




====== RUNNING Conv1D-LSTM ON ALL DATASETS ======



----- Training Conv1D-LSTM for DF1 -----

================ Conv1D-LSTM for DF1 ================
Training samples: 832  | Testing samples: 152


d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 0.2417 - val_loss: 0.0090 - learning_rate: 0.0010
Epoch 2/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0562 - val_loss: 0.0100 - learning_rate: 0.0010
Epoch 3/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0495 - val_loss: 0.0084 - learning_rate: 0.0010
Epoch 4/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0479 - val_loss: 0.0072 - learning_rate: 0.0010
Epoch 5/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0410 - val_loss: 0.0066 - learning_rate: 0.0010
Epoch 6/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0354 - val_loss: 0.0066 - learning_rate: 0.0010
Epoch 7/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0327 - val_loss: 0.0057 - learning_rate: 0.0010
Epoch 8/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0361 - val_loss: 0.0062 - learning_rate: 0.0010
Epoch 9/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0333 - val_loss: 0.0044 - learning_rate: 0.0010
Epoch 10/50
52/52 ━━━━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


119/119 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.1416 - val_loss: 0.0085 - learning_rate: 0.0010
Epoch 2/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0264 - val_loss: 0.0067 - learning_rate: 0.0010
Epoch 3/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0260 - val_loss: 0.0062 - learning_rate: 0.0010
Epoch 4/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0221 - val_loss: 0.0057 - learning_rate: 0.0010
Epoch 5/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0215 - val_loss: 0.0051 - learning_rate: 0.0010
Epoch 6/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0190 - val_loss: 0.0052 - learning_rate: 0.0010
Epoch 7/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0176 - val_loss: 0.0055 - learning_rate: 0.0010
Epoch 8/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0181 - val_loss: 0.0041 - learning_rate: 0.0010
Epoch 9/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0165 - val_loss: 0.0056 - learning_rate: 0.0010
Epoch 10/50
119/119 ━

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 0.3054 - val_loss: 0.1070 - learning_rate: 0.0010
Epoch 2/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0812 - val_loss: 0.0979 - learning_rate: 0.0010
Epoch 3/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0632 - val_loss: 0.1183 - learning_rate: 0.0010
Epoch 4/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0536 - val_loss: 0.1500 - learning_rate: 0.0010
Epoch 5/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0542 - val_loss: 0.1065 - learning_rate: 0.0010
Epoch 6/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0515 - val_loss: 0.0422 - learning_rate: 0.0010
Epoch 7/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0424 - val_loss: 0.0683 - learning_rate: 0.0010
Epoch 8/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0476 - val_loss: 0.0476 - learning_rate: 0.0010
Epoch 9/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0476 - val_loss: 0.1028 - learning_rate: 0.0010
Epoch 10/50
26/26 ━━━━━━━━━━━━━━━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 0.2033 - val_loss: 0.1357 - learning_rate: 0.0010
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0767 - val_loss: 0.0681 - learning_rate: 0.0010
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0751 - val_loss: 0.1003 - learning_rate: 0.0010
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0693 - val_loss: 0.0610 - learning_rate: 0.0010
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0575 - val_loss: 0.0689 - learning_rate: 0.0010
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0536 - val_loss: 0.0173 - learning_rate: 0.0010
Epoch 7/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0446 - val_loss: 0.0331 - learning_rate: 0.0010
Epoch 8/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0432 - val_loss: 0.0168 - learning_rate: 0.0010
Epoch 9/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0374 - val_loss: 0.0422 - learning_rate: 0.0010
Epoch 10/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 

{'DF1': {'High': {'MAE': 0.7923319967169514,
   'RMSE': 0.9784995796572576,
   'R2': 0.9660028518141461,
   'MAPE': 0.995655952117543},
  'Low': {'MAE': 0.8636104182193154,
   'RMSE': 1.0398666306379587,
   'R2': 0.9645113980879267,
   'MAPE': 1.1059815305937186}},
 'DF2': {'High': {'MAE': 0.7474868998808017,
   'RMSE': 1.0273948891870015,
   'R2': 0.9724281009754661,
   'MAPE': 1.2662742005286647},
  'Low': {'MAE': 0.9297587338615868,
   'RMSE': 1.2366945688921682,
   'R2': 0.962329627061171,
   'MAPE': 1.64661576006946}},
 'DF3': {'High': {'MAE': 2.933304234554893,
   'RMSE': 3.5451085769908737,
   'R2': 0.4854702693160541,
   'MAPE': 3.709688890430587},
  'Low': {'MAE': 2.6765262202212683,
   'RMSE': 3.565186630010418,
   'R2': 0.5515979402399189,
   'MAPE': 3.638337417393079}},
 'DF4': {'High': {'MAE': 1.4129276798196035,
   'RMSE': 1.8234243813362,
   'R2': 0.766961332221261,
   'MAPE': 2.195728003591667},
  'Low': {'MAE': 0.9184659670477048,
   'RMSE': 1.3483178240792273,
   'R2'

In [26]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# ---------------- SEQUENCE CREATION ----------------
def create_seq(X, y, seq_len=25):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)


# ---------------- METRICS ----------------
def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true,  y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"\n===== {label} =====")
    print("MAE :",  round(mae, 3))
    print("RMSE:",  round(rmse, 3))
    print("R2  :",  round(r2, 3))
    print("MAPE:", round(mape, 2), "%")

    return mae, rmse, r2, mape



# ---------------- LSTM MODEL ----------------
def build_lstm(input_shape):

    model = Sequential([
        LSTM(128, return_sequences=True, activation='tanh', input_shape=input_shape),
        Dropout(0.2),

        LSTM(64, return_sequences=False, activation='tanh'),
        Dropout(0.2),

        Dense(32, activation='relu'),
        Dense(2)     # Predicting High & Low
    ])

    model.compile(optimizer='adam', loss='mse')
    return model



# ---------------- MAIN LSTM RUN FUNCTION ----------------
def run_LSTM(df, name):

    print(f"\n================ LSTM for {name} ================")

    # Features and target
    X = df[['Open', 'High', 'Low', 'Close']].values
    y = df[['High', 'Low']].values

    seq_len = 25
    train_size = max(int(len(df) * 0.85), seq_len + 3)
    train_size = min(train_size, len(df) - 2)

    X_train_raw = X[:train_size]
    X_test_raw  = X[train_size:]

    y_train_raw = y[:train_size]
    y_test_raw  = y[train_size:]

    # Scaling
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train_raw)
    X_test_scaled  = scaler_X.transform(X_test_raw)

    y_train_scaled = scaler_y.fit_transform(y_train_raw)
    y_test_scaled  = scaler_y.transform(y_test_raw)

    # Merge full scaled data for consistent sequencing
    X_scaled = np.vstack([X_train_scaled, X_test_scaled])
    y_scaled = np.vstack([y_train_scaled, y_test_scaled])

    # Create sequences
    X_seq, y_seq = create_seq(X_scaled, y_scaled, seq_len)

    # Split sequences into train/test
    split = train_size - seq_len
    X_train, X_test = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]

    # Build model
    model = build_lstm((seq_len, X_train_raw.shape[1]))

    # Callbacks
    es = EarlyStopping(patience=6, restore_best_weights=True)
    lr = ReduceLROnPlateau(factor=0.5, patience=3)

    # Train model
    model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=16,
        validation_data=(X_test, y_test) if len(X_test) > 1 else None,
        callbacks=[es, lr],
        verbose=1
    )

    # Predict
    pred_scaled = model.predict(X_test)
    pred = scaler_y.inverse_transform(pred_scaled)
    y_true = scaler_y.inverse_transform(y_test)

    pred_high, pred_low = pred[:, 0], pred[:, 1]
    true_high, true_low = y_true[:, 0], y_true[:, 1]

    m1 = metrics(true_high, pred_high, f"LSTM {name} High")
    m2 = metrics(true_low,  pred_low,  f"LSTM {name} Low")

    return (m1, m2)



# ================= RUN LSTM ON ALL DATASETS ================= #
print("\n\n====== RUNNING LSTM ON ALL DATASETS ======\n")

results_lstm = {}

for df_regime, name in zip([df1, df2, df3, df4], ["DF1", "DF2", "DF3", "DF4"]):
    print(f"\n\n----- Training LSTM for {name} -----")
    try:
        res = run_LSTM(df_regime, name)
        results_lstm[name] = {
            "High": {
                "MAE":  res[0][0],
                "RMSE": res[0][1],
                "R2":   res[0][2],
                "MAPE": res[0][3]
            },
            "Low": {
                "MAE":  res[1][0],
                "RMSE": res[1][1],
                "R2":   res[1][2],
                "MAPE": res[1][3]
            }
        }
    except Exception as e:
        print(f"Error in {name}: {e}")

print("\n\n============= FINAL LSTM RESULTS =============\n")
results_lstm




====== RUNNING LSTM ON ALL DATASETS ======



----- Training LSTM for DF1 -----

================ LSTM for DF1 ================
Epoch 1/50


d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


52/52 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - loss: 0.1321 - val_loss: 0.0122 - learning_rate: 0.0010
Epoch 2/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0407 - val_loss: 0.0100 - learning_rate: 0.0010
Epoch 3/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0386 - val_loss: 0.0100 - learning_rate: 0.0010
Epoch 4/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0315 - val_loss: 0.0094 - learning_rate: 0.0010
Epoch 5/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0303 - val_loss: 0.0097 - learning_rate: 0.0010
Epoch 6/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0288 - val_loss: 0.0090 - learning_rate: 0.0010
Epoch 7/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0300 - val_loss: 0.0068 - learning_rate: 0.0010
Epoch 8/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0243 - val_loss: 0.0072 - learning_rate: 0.0010
Epoch 9/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0243 - val_loss: 0.0069 - learning_rate: 0.0010
Epoch 10/50
52/52 ━━━━━━━━━━━━━━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


119/119 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.0586 - val_loss: 0.0306 - learning_rate: 0.0010
Epoch 2/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0221 - val_loss: 0.0072 - learning_rate: 0.0010
Epoch 3/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0181 - val_loss: 0.0056 - learning_rate: 0.0010
Epoch 4/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0154 - val_loss: 0.0050 - learning_rate: 0.0010
Epoch 5/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0142 - val_loss: 0.0058 - learning_rate: 0.0010
Epoch 6/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0138 - val_loss: 0.0049 - learning_rate: 0.0010
Epoch 7/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0120 - val_loss: 0.0057 - learning_rate: 0.0010
Epoch 8/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0105 - val_loss: 0.0040 - learning_rate: 5.0000e-04
Epoch 9/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0105 - val_loss: 0.0040 - learning_rate: 5.0000e-04
Epoch 10/50


d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - loss: 0.2454 - val_loss: 0.2058 - learning_rate: 0.0010
Epoch 2/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0759 - val_loss: 0.1250 - learning_rate: 0.0010
Epoch 3/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0664 - val_loss: 0.1570 - learning_rate: 0.0010
Epoch 4/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0718 - val_loss: 0.2105 - learning_rate: 0.0010
Epoch 5/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0570 - val_loss: 0.0741 - learning_rate: 0.0010
Epoch 6/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0636 - val_loss: 0.0841 - learning_rate: 0.0010
Epoch 7/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0563 - val_loss: 0.1657 - learning_rate: 0.0010
Epoch 8/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0551 - val_loss: 0.0767 - learning_rate: 0.0010
Epoch 9/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0524 - val_loss: 0.0848 - learning_rate: 5.0000e-04
Epoch 10/50
26/26 ━━━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 0.1842 - val_loss: 0.1050 - learning_rate: 0.0010
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.1038 - val_loss: 0.1447 - learning_rate: 0.0010
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0837 - val_loss: 0.0403 - learning_rate: 0.0010
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0804 - val_loss: 0.0320 - learning_rate: 0.0010
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0675 - val_loss: 0.0314 - learning_rate: 0.0010
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0656 - val_loss: 0.0312 - learning_rate: 0.0010
Epoch 7/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0507 - val_loss: 0.0266 - learning_rate: 0.0010
Epoch 8/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0513 - val_loss: 0.0582 - learning_rate: 0.0010
Epoch 9/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0540 - val_loss: 0.0197 - learning_rate: 0.0010
Epoch 10/50
50/50 ━━━━━━━━━━━━━━━━━

{'DF1': {'High': {'MAE': 0.8979945433767218,
   'RMSE': 1.1088029286415992,
   'R2': 0.9563454083538997,
   'MAPE': 1.124958446140758},
  'Low': {'MAE': 0.9789762496948242,
   'RMSE': 1.1935667035757676,
   'R2': 0.9532451146251978,
   'MAPE': 1.257186785913199}},
 'DF2': {'High': {'MAE': 0.9276851541855756,
   'RMSE': 1.220690913208174,
   'R2': 0.9610772693590707,
   'MAPE': 1.5799292939278482},
  'Low': {'MAE': 1.0869925330666936,
   'RMSE': 1.3996495088644167,
   'R2': 0.951748190462123,
   'MAPE': 1.930479894670676}},
 'DF3': {'High': {'MAE': 2.8692852823357833,
   'RMSE': 3.5443764609808928,
   'R2': 0.48568276298091373,
   'MAPE': 3.645226930755753},
  'Low': {'MAE': 4.104641161466899,
   'RMSE': 4.851518486310216,
   'R2': 0.16965525471422438,
   'MAPE': 5.408184449427581}},
 'DF4': {'High': {'MAE': 1.066226358283056,
   'RMSE': 1.476702901932747,
   'R2': 0.8471594012086134,
   'MAPE': 1.6279267295979543},
  'Low': {'MAE': 0.798198595438918,
   'RMSE': 1.1631291950043323,
   '

In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# ---------------- SEQUENCE CREATION ----------------
def create_seq(X, y, seq_len=25):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)


# ---------------- METRICS ----------------
def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true,  y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"\n===== {label} =====")
    print("MAE :", round(mae, 3))
    print("RMSE:", round(rmse, 3))
    print("R2  :", round(r2, 3))
    print("MAPE:", round(mape, 2), "%")

    return mae, rmse, r2, mape



# ---------------- BiLSTM MODEL ----------------
def build_bilstm(input_shape):

    model = Sequential([

        Bidirectional(LSTM(128, return_sequences=True, activation='tanh'),
                      input_shape=input_shape),
        Dropout(0.2),

        Bidirectional(LSTM(64, return_sequences=False, activation='tanh')),
        Dropout(0.2),

        Dense(32, activation='relu'),
        Dense(2)  # Predicting High & Low
    ])

    model.compile(optimizer='adam', loss='mse')
    return model



# ---------------- MAIN BiLSTM RUN FUNCTION ----------------
def run_BiLSTM(df, name):

    print(f"\n================ BiLSTM for {name} ================")

    # Features and target
    X = df[['Open', 'High', 'Low', 'Close']].values
    y = df[['High', 'Low']].values

    seq_len = 25
    train_size = max(int(len(df) * 0.85), seq_len + 3)
    train_size = min(train_size, len(df) - 2)

    X_train_raw = X[:train_size]
    X_test_raw  = X[train_size:]

    y_train_raw = y[:train_size]
    y_test_raw  = y[train_size:]

    # Scaling
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train_raw)
    X_test_scaled  = scaler_X.transform(X_test_raw)

    y_train_scaled = scaler_y.fit_transform(y_train_raw)
    y_test_scaled  = scaler_y.transform(y_test_raw)

    # Merge full scaled data for consistent sequencing
    X_scaled = np.vstack([X_train_scaled, X_test_scaled])
    y_scaled = np.vstack([y_train_scaled, y_test_scaled])

    # Create sequences
    X_seq, y_seq = create_seq(X_scaled, y_scaled, seq_len)

    # Split sequences
    split = train_size - seq_len
    X_train, X_test = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]

    # Build model
    model = build_bilstm((seq_len, X_train_raw.shape[1]))

    # Callbacks
    es = EarlyStopping(patience=6, restore_best_weights=True)
    lr = ReduceLROnPlateau(factor=0.5, patience=3)

    # Train
    model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=16,
        validation_data=(X_test, y_test) if len(X_test) > 1 else None,
        callbacks=[es, lr],
        verbose=1
    )

    # Predict
    pred_scaled = model.predict(X_test)
    pred = scaler_y.inverse_transform(pred_scaled)
    y_true = scaler_y.inverse_transform(y_test)

    pred_high, pred_low = pred[:, 0], pred[:, 1]
    true_high, true_low = y_true[:, 0], y_true[:, 1]

    m1 = metrics(true_high, pred_high, f"BiLSTM {name} High")
    m2 = metrics(true_low,  pred_low,  f"BiLSTM {name} Low")

    return (m1, m2)



# ================= RUN BiLSTM ON ALL DATASETS ================= #
print("\n\n====== RUNNING BiLSTM ON ALL DATASETS ======\n")

results_bilstm = {}

for df_regime, name in zip([df1, df2, df3, df4], ["DF1", "DF2", "DF3", "DF4"]):
    print(f"\n\n----- Training BiLSTM for {name} -----")
    try:
        res = run_BiLSTM(df_regime, name)
        results_bilstm[name] = {
            "High": {
                "MAE":  res[0][0],
                "RMSE": res[0][1],
                "R2":   res[0][2],
                "MAPE": res[0][3]
            },
            "Low": {
                "MAE":  res[1][0],
                "RMSE": res[1][1],
                "R2":   res[1][2],
                "MAPE": res[1][3]
            }
        }
    except Exception as e:
        print(f"Error in {name}: {e}")

print("\n\n============= FINAL BiLSTM RESULTS =============\n")
results_bilstm




====== RUNNING BiLSTM ON ALL DATASETS ======



----- Training BiLSTM for DF1 -----

================ BiLSTM for DF1 ================
Epoch 1/50


d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


52/52 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - loss: 0.1338 - val_loss: 0.0088 - learning_rate: 0.0010
Epoch 2/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0332 - val_loss: 0.0063 - learning_rate: 0.0010
Epoch 3/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0286 - val_loss: 0.0065 - learning_rate: 0.0010
Epoch 4/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0265 - val_loss: 0.0069 - learning_rate: 0.0010
Epoch 5/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0253 - val_loss: 0.0056 - learning_rate: 0.0010
Epoch 6/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0196 - val_loss: 0.0051 - learning_rate: 0.0010
Epoch 7/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0275 - val_loss: 0.0047 - learning_rate: 0.0010
Epoch 8/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0179 - val_loss: 0.0049 - learning_rate: 0.0010
Epoch 9/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0186 - val_loss: 0.0039 - learning_rate: 0.0010
Epoch 10/50
52/52 ━━━━━━━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


119/119 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - loss: 0.0512 - val_loss: 0.0090 - learning_rate: 0.0010
Epoch 2/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0190 - val_loss: 0.0076 - learning_rate: 0.0010
Epoch 3/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0141 - val_loss: 0.0052 - learning_rate: 0.0010
Epoch 4/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0125 - val_loss: 0.0140 - learning_rate: 0.0010
Epoch 5/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0116 - val_loss: 0.0038 - learning_rate: 0.0010
Epoch 6/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0112 - val_loss: 0.0043 - learning_rate: 0.0010
Epoch 7/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0112 - val_loss: 0.0039 - learning_rate: 0.0010
Epoch 8/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0099 - val_loss: 0.0034 - learning_rate: 0.0010
Epoch 9/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0089 - val_loss: 0.0035 - learning_rate: 0.0010
Epoch 10/50


d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - loss: 0.2435 - val_loss: 0.2931 - learning_rate: 0.0010
Epoch 2/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0834 - val_loss: 0.1203 - learning_rate: 0.0010
Epoch 3/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0709 - val_loss: 0.1987 - learning_rate: 0.0010
Epoch 4/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0599 - val_loss: 0.1060 - learning_rate: 0.0010
Epoch 5/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0504 - val_loss: 0.0924 - learning_rate: 0.0010
Epoch 6/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0585 - val_loss: 0.1998 - learning_rate: 0.0010
Epoch 7/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0488 - val_loss: 0.0554 - learning_rate: 0.0010
Epoch 8/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0458 - val_loss: 0.0706 - learning_rate: 0.0010
Epoch 9/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0503 - val_loss: 0.0904 - learning_rate: 0.0010
Epoch 10/50
26/26 ━━━━━━━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


50/50 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - loss: 0.2183 - val_loss: 0.2011 - learning_rate: 0.0010
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0909 - val_loss: 0.0847 - learning_rate: 0.0010
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0712 - val_loss: 0.1237 - learning_rate: 0.0010
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0666 - val_loss: 0.0443 - learning_rate: 0.0010
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0498 - val_loss: 0.0591 - learning_rate: 0.0010
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0495 - val_loss: 0.0305 - learning_rate: 0.0010
Epoch 7/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0476 - val_loss: 0.0626 - learning_rate: 0.0010
Epoch 8/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0474 - val_loss: 0.0175 - learning_rate: 0.0010
Epoch 9/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0385 - val_loss: 0.0508 - learning_rate: 0.0010
Epoch 10/50
50/50 ━━━━━━━━━━━━

{'DF1': {'High': {'MAE': 0.769410685489052,
   'RMSE': 0.9401677855592814,
   'R2': 0.9686142917219919,
   'MAPE': 0.9627975631510877},
  'Low': {'MAE': 0.8237104415893554,
   'RMSE': 1.0336853243253707,
   'R2': 0.964932055740178,
   'MAPE': 1.0563947402072804}},
 'DF2': {'High': {'MAE': 0.6771574581370634,
   'RMSE': 0.9342342553849705,
   'R2': 0.9772016481323493,
   'MAPE': 1.1558346336162146},
  'Low': {'MAE': 0.8169825497795554,
   'RMSE': 1.0793317481896032,
   'R2': 0.9713064081475801,
   'MAPE': 1.452999532682977}},
 'DF3': {'High': {'MAE': 2.996462771767064,
   'RMSE': 3.5444726365788672,
   'R2': 0.4856548509126166,
   'MAPE': 3.77848875274788},
  'Low': {'MAE': 2.7680588772422388,
   'RMSE': 3.739942652263277,
   'R2': 0.5065615972880497,
   'MAPE': 3.7916389262138734}},
 'DF4': {'High': {'MAE': 1.2628971256621897,
   'RMSE': 1.7588296223750455,
   'R2': 0.7831796622070546,
   'MAPE': 1.9403028437166305},
  'Low': {'MAE': 0.9104208488986917,
   'RMSE': 1.2369136643402168,
 

In [28]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tcn import TCN
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# ---------------- SEQUENCE CREATION ----------------
def create_seq(X, y, seq_len=25):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)


# ---------------- METRICS FUNCTION ----------------
def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true,  y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"\n===== {label} =====")
    print("MAE :", round(mae, 4))
    print("RMSE:", round(rmse, 4))
    print("R2  :", round(r2, 4))
    print("MAPE:", round(mape, 4), "%")

    return mae, rmse, r2, mape


# ---------------- BUILD TCN MODEL ----------------
def build_tcn(input_shape):

    model = Sequential([
        TCN(
            nb_filters=64,
            kernel_size=3,
            dilations=[1, 2, 4, 8],
            dropout_rate=0.2,
            activation='relu',
            return_sequences=False,
            input_shape=input_shape
        ),
        Dense(64, activation='relu'),
        Dense(2)  # Predict High & Low
    ])

    model.compile(optimizer='adam', loss='mse')
    return model


# ---------------- MAIN FUNCTION ----------------
def run_TCN(df, name):

    print(f"\n================ TCN for {name} ================")

    # Features & target
    X = df[['Open', 'High', 'Low', 'Close']].values
    y = df[['High', 'Low']].values

    seq_len = 25
    train_size = max(int(len(df) * 0.85), seq_len + 3)
    train_size = min(train_size, len(df) - 2)

    # Split
    X_train_raw = X[:train_size]
    X_test_raw  = X[train_size:]

    y_train_raw = y[:train_size]
    y_test_raw  = y[train_size:]

    # Scaling
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train_raw)
    X_test_scaled  = scaler_X.transform(X_test_raw)

    y_train_scaled = scaler_y.fit_transform(y_train_raw)
    y_test_scaled  = scaler_y.transform(y_test_raw)

    # Merge for sequences
    X_scaled = np.vstack([X_train_scaled, X_test_scaled])
    y_scaled = np.vstack([y_train_scaled, y_test_scaled])

    # Create sequences
    X_seq, y_seq = create_seq(X_scaled, y_scaled, seq_len)

    split = train_size - seq_len

    X_train, X_test = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]

    # Build TCN model
    model = build_tcn((seq_len, X.shape[1]))

    es = EarlyStopping(patience=6, restore_best_weights=True)
    lr = ReduceLROnPlateau(factor=0.5, patience=3)

    # Train model
    model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=16,
        validation_data=(X_test, y_test) if len(X_test) > 1 else None,
        callbacks=[es, lr],
        verbose=1
    )

    # Predict
    pred_scaled = model.predict(X_test)
    pred = scaler_y.inverse_transform(pred_scaled)
    y_true = scaler_y.inverse_transform(y_test)

    pred_high, pred_low = pred[:, 0], pred[:, 1]
    true_high, true_low = y_true[:, 0], y_true[:, 1]

    m1 = metrics(true_high, pred_high, f"TCN {name} High")
    m2 = metrics(true_low,  pred_low,  f"TCN {name} Low")

    return (m1, m2)



# ------------ RUN TCN ON ALL DATASETS ------------
results_tcn = {}

for df_regime, name in zip([df1, df2, df3, df4], ["DF1", "DF2", "DF3", "DF4"]):
    print(f"\n----- Training TCN for {name} -----")
    try:
        res = run_TCN(df_regime, name)
        results_tcn[name] = {
            "High": {"MAE": res[0][0], "RMSE": res[0][1], "R2": res[0][2], "MAPE": res[0][3]},
            "Low":  {"MAE": res[1][0], "RMSE": res[1][1], "R2": res[1][2], "MAPE": res[1][3]}
        }
    except Exception as e:
        print(f"Error in {name}: {e}")


print("\n\n============= FINAL TCN RESULTS =============")
print(results_tcn)



----- Training TCN for DF1 -----

================ TCN for DF1 ================


d:\Anacondaa\envs\tfenv\lib\site-packages\tcn\tcn.py:268: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super(TCN, self).__init__(**kwargs)



Epoch 1/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 3.2084 - val_loss: 0.0151 - learning_rate: 0.0010
Epoch 2/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.4213 - val_loss: 0.0095 - learning_rate: 0.0010
Epoch 3/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.2733 - val_loss: 0.0072 - learning_rate: 0.0010
Epoch 4/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.2219 - val_loss: 0.0064 - learning_rate: 0.0010
Epoch 5/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1360 - val_loss: 0.0058 - learning_rate: 0.0010
Epoch 6/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1157 - val_loss: 0.0065 - learning_rate: 0.0010
Epoch 7/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1137 - val_loss: 0.0057 - learning_rate: 0.0010
Epoch 8/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1139 - val_loss: 0.0059 - learning_rate: 0.0010
Epoch 9/50
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0939 - val_loss: 0.0057 - learning_rate: 5.0000e-04
Epoch 10/50
52/52 ━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\tcn\tcn.py:268: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super(TCN, self).__init__(**kwargs)


119/119 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 2.1043 - val_loss: 0.0352 - learning_rate: 0.0010
Epoch 2/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.1719 - val_loss: 0.0189 - learning_rate: 0.0010
Epoch 3/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0971 - val_loss: 0.0152 - learning_rate: 0.0010
Epoch 4/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0669 - val_loss: 0.0105 - learning_rate: 0.0010
Epoch 5/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0553 - val_loss: 0.0105 - learning_rate: 0.0010
Epoch 6/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0452 - val_loss: 0.0081 - learning_rate: 0.0010
Epoch 7/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0385 - val_loss: 0.0092 - learning_rate: 0.0010
Epoch 8/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0323 - val_loss: 0.0084 - learning_rate: 0.0010
Epoch 9/50
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0293 - val_loss: 0.0079 - learning_rate: 0.0010
Epoch 10/50
119/119 ━

d:\Anacondaa\envs\tfenv\lib\site-packages\tcn\tcn.py:268: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super(TCN, self).__init__(**kwargs)


26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - loss: 7.0146 - val_loss: 0.9193 - learning_rate: 0.0010
Epoch 2/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 1.0580 - val_loss: 0.3910 - learning_rate: 0.0010
Epoch 3/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.6794 - val_loss: 0.2636 - learning_rate: 0.0010
Epoch 4/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.3983 - val_loss: 0.1202 - learning_rate: 0.0010
Epoch 5/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.3425 - val_loss: 0.1390 - learning_rate: 0.0010
Epoch 6/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.3136 - val_loss: 0.1730 - learning_rate: 0.0010
Epoch 7/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.2735 - val_loss: 0.0786 - learning_rate: 0.0010
Epoch 8/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.2025 - val_loss: 0.1592 - learning_rate: 0.0010
Epoch 9/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.2033 - val_loss: 0.0723 - learning_rate: 0.0010
Epoch 10/50
26/26 ━━━━━━━━━━━━━━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\tcn\tcn.py:268: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super(TCN, self).__init__(**kwargs)


50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - loss: 5.1108 - val_loss: 0.2847 - learning_rate: 0.0010
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.5506 - val_loss: 0.1675 - learning_rate: 0.0010
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.3909 - val_loss: 0.0475 - learning_rate: 0.0010
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.2839 - val_loss: 0.1595 - learning_rate: 0.0010
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.2400 - val_loss: 0.0860 - learning_rate: 0.0010
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1955 - val_loss: 0.0848 - learning_rate: 0.0010
Epoch 7/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1843 - val_loss: 0.0586 - learning_rate: 5.0000e-04
Epoch 8/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1733 - val_loss: 0.0541 - learning_rate: 5.0000e-04
Epoch 9/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1691 - val_loss: 0.0462 - learning_rate: 5.0000e-04
Epoch 10/50
50/50 ━━━━━━━━

In [17]:
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis, jarque_bera
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt


In [18]:
def get_stats(df, target="Close"):
    s = df[target].dropna()

    stats = {}

    # Basic statistics
    stats["Mean"] = s.mean()
    stats["Std"]  = s.std()
    stats["CV"]   = s.std() / s.mean()

    # Distribution shape
    stats["Skewness"] = skew(s)
    stats["Kurtosis"] = kurtosis(s, fisher=True)

    # Normality
    stats["JB_pvalue"] = jarque_bera(s)[1]

    # Stationarity
    stats["ADF_pvalue"] = adfuller(s)[1]

    try:
        stats["KPSS_pvalue"] = kpss(s, regression="c")[1]
    except:
        stats["KPSS_pvalue"] = np.nan

    return stats


In [19]:
results = pd.DataFrame({
    "FULL_DATA": get_stats(df),
    "DF1": get_stats(df1),
    "DF2": get_stats(df2),
    "DF3": get_stats(df3),
    "DF4": get_stats(df4)
})

results = results.round(4)
results


C:\Users\ASUS\AppData\Local\Temp\ipykernel_27748\995538300.py:22: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  stats["KPSS_pvalue"] = kpss(s, regression="c")[1]
C:\Users\ASUS\AppData\Local\Temp\ipykernel_27748\995538300.py:22: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  stats["KPSS_pvalue"] = kpss(s, regression="c")[1]
C:\Users\ASUS\AppData\Local\Temp\ipykernel_27748\995538300.py:22: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  stats["KPSS_pvalue"] = kpss(s, regression="c")[1]
C:\Users\ASUS\AppData\Local\Temp\ipykernel_27748\995538300.py:22: InterpolationWarning: The test statistic is outside of the range of p-values av

,FULL_DATA,DF1,DF2,DF3,DF4
Mean,72.6861,78.4761,71.7197,53.6967,78.8034
Std,21.7285,22.0020,22.8889,17.4712,12.8324
CV,0.2989,0.2804,0.3191,0.3254,0.1628
Skewness,0.1649,0.6980,0.1272,-0.4574,1.0522
Kurtosis,-0.2604,0.5655,-1.4844,0.4390,1.0590
JB_pvalue,0.0000,0.0000,0.0000,0.0000,0.0000
ADF_pvalue,0.0327,0.5238,0.5894,0.8359,0.6475
KPSS_pvalue,0.0100,0.0653,0.0100,0.0100,0.0100
